# Full match: pipeline on every 5-minute segment, then players joined across the match

Input: the match's playing-time segments, `MyDrive/Playbook/video_downloads/match_video_2_clips/*.mp4`
(01_1H_00-05 ... 20_2H_90-94). Output, in `MyDrive/Playbook/full_match/match_video_2/`:
- `match_tracks.csv.gz`: every frame of the match on one timeline (`frame`), with `segment` /
  `segment_frame`, teams labelled the same in every segment (team 0 / 1), and `joined_id`: the
  tracker's pieces joined into players by the appearance model (never two players on screen together,
  never across teams). This is the file `tag_passes_v2` opens.
- `match.mp4`: the segments back to back, frame for frame the same timeline.
- `preview_joined.mp4`: the first 3 minutes with joined player IDs, to look at.
- `summary.json`: speed per segment, pieces, players joined, team alignment.

Settings: `baseline.env` (fp16, keypoints and team colours every 3rd frame), people model as a
TensorRT engine, ball model fp16.

**Runtime:** L4 GPU, about 6-7 hours for the whole match (each segment ~20 minutes), then ~30 minutes
for joining and the videos. **Runtime -> Run all**; do not edit cells. Keep the tab open (Colab stops
runtimes whose tab is closed for a while). If it disconnects, just **Run all again**: finished segments
are skipped, only the segment that was running starts over.

In [ ]:
# Cell 1 — GPU, Drive, inputs
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
GPU = g.stdout.strip(); print('GPU:', GPU)
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
if not os.path.isdir(PBD):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
MATCH = 'match_video_2'
CLIPS = sorted(glob.glob(f'{PBD}/video_downloads/{MATCH}_clips/*.mp4'))
OUT = f'{PBD}/full_match/{MATCH}'; SEG = f'{OUT}/segments'; os.makedirs(SEG, exist_ok=True)
MODEL = f'{PBD}/reid_appearance/appearance_model.pt'
NEEDED = [f'{PBD}/people_model/train/checkpoint_best_total.pth', f'{PBD}/ball_model/train/checkpoint_best_total.pth',
          f'{PBD}/field_model/roboflow_v10/weights.pt', MODEL]
missing = [p for p in NEEDED if not os.path.exists(p)]
if missing or not CLIPS:
    raise SystemExit(f'Missing in Drive: {missing or "the match clips"}')
NAMES = [os.path.basename(c)[:-4] for c in CLIPS]
print(len(CLIPS), 'segments:', NAMES[0], '...', NAMES[-1])

In [ ]:
# Cell 2 — Our repo and packages (TensorRT included)
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
!pip install -q "rfdetr[train,tensorrt]==1.11.0" "git+https://github.com/roboflow/sports.git@main" "pydantic-settings==2.4.0" "python-dotenv==1.0.1" "lap>=0.5.13,<0.6" "transformers>=4.47"
shutil.copy('baseline.env', '.env')
import numpy as np, pandas as pd
print('ready')

In [ ]:
# Cell 3 — The pipeline on each segment (skips finished ones; ~20 min each)
ACCEL = {'MODEL_ACCEL': 'tensorrt', 'BALL_MODEL_ACCEL': 'fp16'}
STATE = f'{OUT}/run_settings.json'
if os.path.exists(STATE):
    ACCEL = json.load(open(STATE))['accel']
done_minutes = []
for i, (clip, name) in enumerate(zip(CLIPS, NAMES)):
    dst = f'{SEG}/{name}'
    if os.path.exists(f'{dst}/kpi_summary.json'):
        continue
    local = f'/content/seg_{name}'
    shutil.rmtree(local, ignore_errors=True)
    t0 = time.time()
    run = lambda accel: subprocess.run([sys.executable, 'main.py', '--source-video', clip, '--out-dir', local,
                                        '--no-video', '--enable-team'], capture_output=True, text=True,
                                       env={**os.environ, **accel})
    r = run(ACCEL)
    if r.returncode != 0 and ACCEL['MODEL_ACCEL'] == 'tensorrt':
        print('TensorRT failed, using fp16 for everything:', r.stderr[-800:])
        ACCEL = {'MODEL_ACCEL': 'fp16', 'BALL_MODEL_ACCEL': 'fp16'}
        r = run(ACCEL)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-3000:])
        raise SystemExit(f'{name} failed: send Claude a screenshot of the lines above.')
    json.dump({'accel': ACCEL, 'gpu': GPU}, open(STATE, 'w'))
    os.makedirs(dst, exist_ok=True)
    pd.read_csv(f'{local}/per_frame_tracks.csv', low_memory=False).to_csv(f'{dst}/per_frame_tracks.csv.gz', index=False, compression='gzip')
    shutil.copy(f'{local}/kpi_summary.json', f'{dst}/kpi_summary.json')
    mins = (time.time() - t0) / 60
    json.dump({'minutes': round(mins, 1), 'gpu': GPU, 'accel': ACCEL}, open(f'{dst}/run_time.json', 'w'))
    done_minutes.append(mins)
    left = sum(not os.path.exists(f'{SEG}/{n}/kpi_summary.json') for n in NAMES)
    kpi = json.load(open(f'{dst}/kpi_summary.json'))
    print(f'{name}: {mins:.0f} min, {kpi["effective_fps"]:.1f} frames/s, ball {kpi["ball_detect_coverage"]:.0%}, '
          f'pitch {kpi["homography_ok_rate"]:.0%}  |  {left} segments left, about {left * np.mean(done_minutes) / 60:.1f} h')
print('all segments done')

In [ ]:
# Cell 4 — Appearance of every tracker piece (skips finished segments; ~1 min each)
from tools.reid_appearance import Embedder, crops_from_video, sample_rows
import torch
emb_model = None
for clip, name in zip(CLIPS, NAMES):
    f = f'{SEG}/{name}/pieces_emb.npz'
    if os.path.exists(f):
        continue
    if emb_model is None:
        emb_model = Embedder(); emb_model.load_state_dict(torch.load(MODEL, map_location=emb_model.device))
    t = pd.read_csv(f'{SEG}/{name}/per_frame_tracks.csv.gz', low_memory=False)
    t = t[t.class_id.isin((1, 2)) & (t.raw_tracker_id >= 0)]
    crops, rows = crops_from_video(clip, sample_rows(t, 'raw_tracker_id', per_id=12))
    np.savez_compressed(f, emb=emb_model.embed(crops).astype(np.float16), piece=rows.raw_tracker_id.to_numpy())
    print(name, len(crops), 'crops')
print('appearance done')

In [ ]:
# Cell 5 — One match timeline, teams aligned, pieces joined into players
from tools.full_match import ID_STRIDE, merge_segments, team_mapping, apply_team_mapping
from tools.join_pieces import piece_table, cluster, joined_ids
FINAL = f'{OUT}/match_tracks.csv.gz'
segs, embs, crop_piece = [], [], []
for i, name in enumerate(NAMES):
    t = pd.read_csv(f'{SEG}/{name}/per_frame_tracks.csv.gz', low_memory=False)
    n = json.load(open(f'{SEG}/{name}/kpi_summary.json'))['processed_frames']
    segs.append((name, t, n))
    d = np.load(f'{SEG}/{name}/pieces_emb.npz')
    embs.append(d['emb'].astype(np.float32)); crop_piece.append(d['piece'] + i * ID_STRIDE)
match = merge_segments(segs); del segs
emb, crop_piece = np.concatenate(embs), np.concatenate(crop_piece)
pieces = piece_table(match)
seg_of = match[match.raw_tracker_id >= 0].drop_duplicates('raw_tracker_id').set_index('raw_tracker_id').segment.to_dict()
maps = team_mapping(emb, crop_piece, {p: v['team'] for p, v in pieces.items()}, seg_of, NAMES)
match = apply_team_mapping(match, maps)
pieces = piece_table(match)
t0 = time.time()
lab = cluster(pieces, emb, crop_piece, thr=0.3)
match['joined_id'] = joined_ids(match, lab)
people = match.class_id.isin((1, 2)) & (match.joined_id > 0)
match.loc[people, 'display_track_id'] = match.loc[people, 'joined_id']   # what the tagging tool shows before a name is typed
match.to_csv(FINAL, index=False, compression='gzip')
seg_stats = {n: {k: json.load(open(f'{SEG}/{n}/{f}')).get(k) for f, ks in (('kpi_summary.json', ('processed_frames', 'effective_fps', 'ball_detect_coverage', 'homography_ok_rate')), ('run_time.json', ('minutes',))) for k in ks} for n in NAMES}
summary = {'segments': seg_stats, 'frames': int(match.frame.max() + 1), 'pieces': len(pieces),
           'players_joined': int(match.joined_id[people].nunique()), 'join_seconds': round(time.time() - t0),
           'team_mapping': {n: {str(k): v for k, v in m.items()} for n, m in maps.items()},
           'pipeline_hours': round(sum(s['minutes'] or 0 for s in seg_stats.values()) / 60, 2)}
json.dump(summary, open(f'{OUT}/summary.json', 'w'), indent=1)
print({k: v for k, v in summary.items() if k not in ('segments', 'team_mapping')})
print('team alignment margins (bigger = clearer):', {n: m['margin'] for n, m in maps.items()})

In [ ]:
# Cell 6 — The segments back to back as one video (same frames as match_tracks.csv.gz)
import cv2
VIDEO = f'{OUT}/match.mp4'
total = sum(json.load(open(f'{SEG}/{n}/kpi_summary.json'))['processed_frames'] for n in NAMES)
def frames_in(p):
    cap = cv2.VideoCapture(p); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release(); return n
if not os.path.exists(VIDEO) or frames_in(VIDEO) != total:
    open('/content/clips.txt', 'w').write(''.join(f"file '{c}'\n" for c in CLIPS))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'concat', '-safe', '0', '-i', '/content/clips.txt',
                    '-c', 'copy', '-movflags', '+faststart', '/content/match.mp4'], check=True)
    n = frames_in('/content/match.mp4')
    if n != total:   # stream copy changed the frame count at a joint: re-encode (slower, exact)
        print(f'joined video has {n} frames, tracks have {total}: re-encoding ...')
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'concat', '-safe', '0', '-i', '/content/clips.txt',
                        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20', '-an', '-vsync', '0',
                        '-movflags', '+faststart', '/content/match.mp4'], check=True)
        n = frames_in('/content/match.mp4')
    shutil.copy('/content/match.mp4', VIDEO)
    print('video frames:', n, '| tracks frames:', total, '| OK' if n == total else '| MISMATCH: tell Claude')
else:
    print('match video already there')

In [ ]:
# Cell 7 — Preview: the first 3 minutes with joined player IDs (team colours)
from tools.render_run import render
PREVIEW = f'{OUT}/preview_joined.mp4'
if not os.path.exists(PREVIEW):
    first = pd.read_csv(FINAL, low_memory=False)
    first = first[first.frame < 3 * 60 * 30]
    first.to_csv('/content/preview.csv', index=False)
    render(VIDEO, {'joined players': '/content/preview.csv'}, '/content/preview.mp4', width=1280,
           frames=range(0, 3 * 60 * 30), id_col='joined_id', team_colors=True)
    shutil.copy('/content/preview.mp4', PREVIEW)
print('preview:', PREVIEW)
print(f'\nDone. Tell Claude the results are in {OUT}. For tagging: notebooks/tag_passes_v2.ipynb opens this match.')